# Property analysis — single property deep dive

Pick a property + scenario YAML, run the full rent-vs-buy analysis.

**How to use:**
1. Edit `PROPERTY` and `SCENARIO` below to point at the YAMLs you want
2. Run all cells
3. Read the verdict, sensitivity, and rent-out sections

Re-run with different scenarios (`base.yaml`, `optimistic.yaml`, `pessimistic.yaml`) to see how assumptions move the answer.

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from mortgage_calc import (
    amortization_schedule, breakeven_appreciation, breakeven_hold_period,
    load_property, load_scenario, run_rent_out, run_scenario,
    sweep_2d, tornado, Loan,
)

# --- CONFIGURE HERE ---
PROPERTY = "../properties/sample_lincoln_park.yaml"
SCENARIO = "../scenarios/base.yaml"
# ----------------------

prop = load_property(PROPERTY)
s = load_scenario(SCENARIO)
buy, rent, ctx, rent_out = s["buy"], s["rent"], s["ctx"], s["rent_out"]

print(f"Property: {prop.name}")
print(f"Price: ${prop.price:,.0f}")
print(f"HOA: ${prop.monthly_hoa}/mo  |  Tax: ${prop.monthly_property_tax}/mo")
print(f"\nScenario: {SCENARIO}")
print(f"Down: {buy.down_payment_pct*100:.0f}%  |  Rate: {buy.mortgage_rate*100:.2f}%")
print(f"Hold: {ctx.hold_period_years} yrs  |  Appreciation: {buy.annual_appreciation*100:.1f}%/yr")
print(f"Equivalent rent: ${rent.monthly_rent}/mo")

## 1. The verdict

In [ ]:
result = run_scenario(prop, buy, rent, ctx)

verdict = "BUYING WINS" if result.true_economic_gain > 0 else "RENTING WINS"
print(f"=== {verdict} by ${abs(result.true_economic_gain):,.0f} ===\n")

print(f"Cash invested:           ${result.cash_invested:>12,.0f}")
print(f"Monthly cost (yr 1):     ${result.monthly_payment_year_1:>12,.0f}")
print(f"vs. equivalent rent:     ${rent.monthly_rent:>12,.0f}")
print()
print(f"Sale price ({ctx.hold_period_years} yrs):     ${result.sale_price:>12,.0f}")
print(f"Selling costs (6%):     -${result.selling_costs:>12,.0f}")
print(f"Mortgage payoff:        -${result.mortgage_payoff:>12,.0f}")
print(f"Net cash from sale:      ${result.net_cash_from_sale:>12,.0f}")
print(f"Less cash invested:     -${result.cash_invested:>12,.0f}")
print(f"Profit on sale:          ${result.profit_on_sale:>12,.0f} ({result.cash_on_cash_return_pct:.1f}% on cash)")
print()
print(f"+ Rent savings over hold: ${(result.total_rent_paid - result.total_paid_owning):>11,.0f}")
print(f"- Opportunity cost (7%):  ${result.opportunity_cost_of_cash:>11,.0f}")
if result.tax_benefit > 0:
    print(f"+ Tax benefit:            ${result.tax_benefit:>11,.0f}")
print(f"= True economic gain:    ${result.true_economic_gain:>12,.0f}")

## 2. Amortization — how does equity build over time?

In [ ]:
loan = Loan(
    principal=prop.price * (1 - buy.down_payment_pct),
    annual_rate=buy.mortgage_rate,
    term_years=buy.mortgage_term_years,
)
amort = amortization_schedule(loan)
amort["year"] = (amort["month"] - 1) // 12 + 1

hold_amort = amort[amort["month"] <= ctx.hold_period_years * 12]

fig, ax = plt.subplots(figsize=(10, 5))
yearly = hold_amort.groupby("year").agg(
    interest=("interest", "sum"),
    principal=("principal", "sum"),
)
yearly.plot(kind="bar", stacked=True, ax=ax, color=["#D4543C", "#3B6D11"])
ax.set_title(f"Where your mortgage dollars go (first {ctx.hold_period_years} years)")
ax.set_ylabel("$ per year")
ax.legend(["Interest (gone)", "Principal (builds equity)"])
plt.tight_layout()
plt.show()

print(f"Total interest paid over {ctx.hold_period_years} yrs: ${result.interest_paid:,.0f}")
print(f"Total principal paid (= equity built): ${result.principal_paid:,.0f}")
print(f"Ratio principal/interest: {result.principal_paid / result.interest_paid:.2f}")

## 3. Sensitivity — which inputs actually matter?

Tornado chart: each bar shows how much the bottom line moves when one input swings between reasonable extremes.

In [ ]:
t = tornado(prop, buy, rent, ctx)
display(t)

fig, ax = plt.subplots(figsize=(10, 5))
y = np.arange(len(t))
ax.barh(y, t["high_output"] - t["baseline_output"], left=t["baseline_output"], color="#1D9E75", label="high input")
ax.barh(y, t["low_output"] - t["baseline_output"], left=t["baseline_output"], color="#D85A30", label="low input")
ax.axvline(t.iloc[0]["baseline_output"], color="black", linewidth=1, linestyle="--", label="baseline")
ax.set_yticks(y)
ax.set_yticklabels(t["param"])
ax.invert_yaxis()
ax.set_xlabel("True economic gain ($)")
ax.set_title("Tornado: which inputs move the answer most")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Appreciation × hold period — the 2D heatmap

The two most uncertain inputs. Green = buy wins, red = rent wins.

In [ ]:
appreciation_grid = np.arange(-0.02, 0.081, 0.01)
hold_grid = np.arange(2, 16)

heat = sweep_2d(
    prop, buy, rent, ctx,
    param_x="ctx.hold_period_years", values_x=hold_grid,
    param_y="buy.annual_appreciation", values_y=appreciation_grid,
)

heat.index = [f"{x*100:.0f}%" for x in heat.index]
heat.columns = [f"{int(c)}yr" for c in heat.columns]

fig, ax = plt.subplots(figsize=(11, 6))
sns.heatmap(
    heat / 1000,
    cmap="RdYlGn",
    center=0,
    annot=True,
    fmt=".0f",
    cbar_kws={"label": "True gain ($K)"},
    ax=ax,
)
ax.set_title(f"{prop.name}: buy-vs-rent gain ($K) by appreciation × hold period")
ax.set_xlabel("Hold period")
ax.set_ylabel("Annual appreciation")
plt.tight_layout()
plt.show()

## 5. Breakeven points

In [ ]:
be_app = breakeven_appreciation(prop, buy, rent, ctx)
be_hold = breakeven_hold_period(prop, buy, rent, ctx)

print(f"At current hold period ({ctx.hold_period_years} yrs):")
if be_app is not None:
    print(f"  Need appreciation > {be_app*100:.2f}%/yr for buying to win")
else:
    print(f"  No breakeven exists in the searched range")

print(f"\nAt current appreciation ({buy.annual_appreciation*100:.1f}%/yr):")
if be_hold is not None:
    print(f"  Need to hold {be_hold}+ years for buying to win")
else:
    print(f"  Buying never wins within 20 years at this appreciation")

## 6. Rent-out scenario — what if you relocate?

Instead of selling, keep the property and rent it out.

In [ ]:
if rent_out is None:
    print("No rent_out assumptions in this scenario.")
else:
    ro_result = run_rent_out(prop, buy, rent_out, years=ctx.hold_period_years)

    yearly_df = pd.DataFrame([
        {
            "year": y.year,
            "effective_rent": y.effective_rent,
            "mortgage_pi": y.mortgage_pi,
            "all_other_costs": y.hoa + y.property_tax + y.insurance + y.pmi + y.property_mgmt + y.maintenance,
            "net_cash_flow": y.net_cash_flow,
            "principal_built": y.principal_paid,
        }
        for y in ro_result.yearly
    ])
    display(yearly_df.round(0))

    print(f"\nCumulative net cash flow over {ctx.hold_period_years} yrs: ${ro_result.cumulative_cash_flow:,.0f}")
    print(f"Cumulative principal paid:    ${ro_result.cumulative_principal_paid:,.0f}")
    print(f"Final loan balance:           ${ro_result.final_balance:,.0f}")
    print(f"Final property value:         ${ro_result.final_property_value:,.0f}")

    annual_cf = ro_result.cumulative_cash_flow / ctx.hold_period_years
    if annual_cf < 0:
        print(f"\n⚠️  Average annual cash flow is NEGATIVE: ${annual_cf:,.0f}/yr")
        print(f"   You'd be subsidizing ~${abs(annual_cf)/12:,.0f}/mo from elsewhere.")
    else:
        print(f"\n✓ Average annual cash flow: ${annual_cf:,.0f} (${annual_cf/12:,.0f}/mo)")

## 7. Iterate

To run a different property or scenario, edit `PROPERTY` and `SCENARIO` at the top and re-run.

For multi-property comparison, see `02_multi_property_comparison.ipynb`.